In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
batch_size = 4
seq_len = 64
embed_dim = 128

x = torch.randn(batch_size, seq_len, embed_dim)
print("Input shape:", x.shape)

Input shape: torch.Size([4, 64, 128])


## Implement Attention without any learnable parameters

In [5]:
## First compute the attention scores
similarity = (x @ x.transpose(-2, -1))

print(f"Before softmax, similarity variance: {similarity.var()}") # to high variance
similarity_norm = similarity / (embed_dim ** 0.5)
print(f"After normalization, similarity variance: {similarity_norm.var()}")

print("Similarity shape:", similarity.shape)

## Now apply softmax to get the attention weights
attention_weights = F.softmax(similarity_norm, dim=-1)

## Now compute the attention output
context_vectors = attention_weights @ x
print("Context vectors shape:", context_vectors.shape)

Before softmax, similarity variance: 387.61871337890625
After normalization, similarity variance: 3.028271198272705
Similarity shape: torch.Size([4, 64, 64])
Context vectors shape: torch.Size([4, 64, 128])


## Add Learnable Parameters

In [10]:
# Create a linear layer that projects from 10 dimensions to 20 dimensions
linear = nn.Linear(10, 20)

# Create a random tensor with shape (batch_size=4, sequence_length=5, feature_dim=10)
rand = torch.randn(4, 5, 10)
print("Input to linear layer shape:", rand.shape)

# Apply the linear transformation
output = linear(rand)
print("Output shape after linear layer:", output.shape)

# Detailed explanation of how linear layers work with multi-dimensional tensors:
# 1. The linear layer only operates on the LAST dimension of the input tensor
# 2. All other dimensions are preserved and treated as "batch dimensions"
# 3. Internally, PyTorch reshapes the input from (4, 5, 10) to (4*5, 10) = (20, 10)
# 4. Then applies the linear transformation: (20, 10) @ (10, 20) + bias = (20, 20)
# 5. Finally reshapes back to (4, 5, 20)
# 
# This is equivalent to:
# rand_reshaped = rand.view(-1, 10)  # Shape: (20, 10)
# output_reshaped = linear(rand_reshaped)  # Shape: (20, 20)
# output = output_reshaped.view(4, 5, 20)  # Shape: (4, 5, 20)
#
# This behavior makes linear layers very convenient for:
# - Processing sequences (batch_size, seq_len, features)
# - Processing images after flattening (batch_size, height, width, channels)
# - Any tensor where you want to transform only the feature dimension

Input to linear layer shape: torch.Size([4, 5, 10])
Output shape after linear layer: torch.Size([4, 5, 20])


In [7]:
class Attention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.embed_dim = embed_dim
        self.scale = embed_dim ** 0.5
        
        # create pointwise projection layers
        self.query_proj = nn.Linear(embed_dim, embed_dim)
        self.key_proj = nn.Linear(embed_dim, embed_dim)
        self.value_proj = nn.Linear(embed_dim, embed_dim)
        
    def forward(self, x):
        # compute query, key, value
        query = self.query_proj(x)
        key = self.key_proj(x)
        value = self.value_proj(x)
        
        # compute attention scores
        similarity = (query @ key.transpose(-2, -1)) / self.scale
        
        # apply softmax to get attention weights
        attention_weights = F.softmax(similarity, dim=-1)
        
        # compute context vectors
        context_vectors = attention_weights @ value
        
        return context_vectors

In [8]:
attention = Attention(embed_dim = 128)
context_vectors = attention(x)
print("Context vectors shape with learnable parameters:", context_vectors.shape)

Context vectors shape with learnable parameters: torch.Size([4, 64, 128])


## MultiHeaded Attention


In [12]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        assert self.head_dim * num_heads == embed_dim, "embed_dim must be divisible by num_heads"
        
        # create pointwise projection layers for queries, keys, values
        self.query_proj = nn.Linear(embed_dim, embed_dim)
        self.key_proj = nn.Linear(embed_dim, embed_dim)
        self.value_proj = nn.Linear(embed_dim, embed_dim)
        
        # output projection layer
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        
    def forward(self, x):
        batch_size, seq_len, _ = x.shape  # x: (4, 64, 128)
        
        # Step 1: Project input to queries, keys, values
        query = self.query_proj(x)  # (4, 64, 128) -> (4, 64, 128)
        key = self.key_proj(x)      # (4, 64, 128) -> (4, 64, 128)
        value = self.value_proj(x)  # (4, 64, 128) -> (4, 64, 128)
        
        # Step 2: Reshape for multi-head attention
        # Split embed_dim into (num_heads, head_dim)
        query = query.view(batch_size, seq_len, self.num_heads, self.head_dim)  # (4, 64, 8, 16)
        key = key.view(batch_size, seq_len, self.num_heads, self.head_dim)      # (4, 64, 8, 16)
        value = value.view(batch_size, seq_len, self.num_heads, self.head_dim)  # (4, 64, 8, 16)
        
        # Step 3: Transpose to get heads as separate batch dimension
        # Move num_heads dimension to position 1 for parallel processing
        query = query.transpose(1, 2)  # (4, 64, 8, 16) -> (4, 8, 64, 16)
        key = key.transpose(1, 2)      # (4, 64, 8, 16) -> (4, 8, 64, 16)
        value = value.transpose(1, 2)  # (4, 64, 8, 16) -> (4, 8, 64, 16)
        
        # Step 4: Compute attention scores for all heads simultaneously
        # Each head operates on head_dim=16 dimensions instead of full embed_dim=128
        similarity = (query @ key.transpose(-2, -1)) / (self.head_dim ** 0.5)
        # (4, 8, 64, 16) @ (4, 8, 16, 64) -> (4, 8, 64, 64)
        # Scale by sqrt(head_dim) = sqrt(16) = 4
        
        # Step 5: Apply softmax to get attention weights
        attention_weights = F.softmax(similarity, dim=-1)  # (4, 8, 64, 64)
        
        # Step 6: Compute context vectors for each head
        context_vectors = attention_weights @ value  # (4, 8, 64, 64) @ (4, 8, 64, 16) -> (4, 8, 64, 16)
        
        # Step 7: Concatenate heads back together
        # First transpose back: (4, 8, 64, 16) -> (4, 64, 8, 16)
        context_vectors = context_vectors.transpose(1, 2)
        # Then reshape to merge heads: (4, 64, 8, 16) -> (4, 64, 128)
        context_vectors = context_vectors.contiguous().view(batch_size, seq_len, -1)
        
        # Step 8: Apply final output projection
        output = self.out_proj(context_vectors)  # (4, 64, 128) -> (4, 64, 128)
        
        return output

In [13]:
embed_dim = 128
num_heads = 8
multi_head_attention = MultiHeadAttention(embed_dim, num_heads)
context_vectors = multi_head_attention(x)
print("Context vectors shape with multi-head attention:", context_vectors.shape)

Context vectors shape with multi-head attention: torch.Size([4, 64, 128])


In [14]:
seq_len = 64
x = torch.randn(batch_size, seq_len, embed_dim)
print("Input shape:", x.shape)
output = multi_head_attention(x)
print("Output shape after multi-head attention:", output.shape)

Input shape: torch.Size([4, 64, 128])
Output shape after multi-head attention: torch.Size([4, 64, 128])
